<a href="https://colab.research.google.com/github/DmitryDol/BigData_Dolzhikov_6403/blob/main/L2%20-%20Reports%20with%20Apache%20Spark/BD_lab2_%D0%94%D0%BE%D0%BB%D0%B6%D0%B8%D0%BA%D0%BE%D0%B2%D0%94%D0%9F_6403.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, substring, split, explode, regexp_replace


In [3]:
spark = SparkSession.builder \
    .appName("StackExchange_XML") \
    .master("local[*]") \
    .getOrCreate()

In [4]:
posts_df = spark.read.format("xml") \
    .option("rowTag", "row") \
    .load("/content/drive/MyDrive/Colab Notebooks/BigData/data/posts_sample.xml")

posts_df.show(5)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|_Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+-

In [5]:
posts_filtered = posts_df.select(
    substring(col("_CreationDate"), 1, 4).cast("int").alias("Year"),
    col("_Tags").alias("RawTags")
).filter("Year BETWEEN 2010 AND 2020 AND RawTags IS NOT NULL")

posts_exploded = posts_filtered.withColumn("Tag", explode(split(regexp_replace(lower(col("RawTags")), "^<|>$", ""), "><")))

In [6]:
languages_list = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/BigData/data/programming-languages.csv", header=True).withColumn("LangName", lower(col("name")))

In [7]:
posts_exploded.createOrReplaceTempView("posts_tags")
languages_list.createOrReplaceTempView("languages")

top_10_report = spark.sql("""
    with LanguageCounts as (
        select
            p.Year,
            p.Tag as Language,
            count(*) as Count
        from posts_tags p
        join languages l ON p.Tag = l.LangName
        group by p.Year, p.Tag
    ),
    RankedLanguages as (
        select
            Year,
            Language,
            Count,
            row_number() over (partition by Year order by Count desc) as Rank
        from LanguageCounts
    )
    select Year, Language, Count
    from RankedLanguages
    where Rank <= 10
    order by Year desc, Count desc
""")

top_10_report.show(20)

+----+----------+-----+
|Year|  Language|Count|
+----+----------+-----+
|2019|    python|  166|
|2019|javascript|  135|
|2019|      java|   95|
|2019|       php|   65|
|2019|         r|   37|
|2019|typescript|   17|
|2019|         c|   14|
|2019|      bash|   11|
|2019|        go|    9|
|2019|    matlab|    9|
|2018|    python|  220|
|2018|javascript|  198|
|2018|      java|  146|
|2018|       php|  111|
|2018|         r|   66|
|2018|typescript|   27|
|2018|         c|   24|
|2018|     scala|   23|
|2018|powershell|   13|
|2018|      bash|   12|
+----+----------+-----+
only showing top 20 rows


In [8]:
top_10_report.write.parquet("/content/drive/MyDrive/Colab Notebooks/BigData/L2 - Reports with Apache Spark/top_10_languages.parquet")